In [1]:
!pip install opencv-python

   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   -----------------

In [15]:
!pip install langchain langchain-community

In [17]:
!pip install openai fastapi uvicorn pydantic

In [1]:
import os
import json
import cv2
import numpy as np
import threading
import time
from fastapi import FastAPI, HTTPException
from fastapi.responses import HTMLResponse
from pydantic import BaseModel
from openai import OpenAI
import uvicorn

# --- CORRECTIF INDISPENSABLE POUR NOTEBOOK JUPYTER ---
import nest_asyncio
nest_asyncio.apply()

app = FastAPI(title="Agent IA Plateforme + Caméra Temps Réel")

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "TA_CLE_API_ICI"))

def logique_flux_camera():
    print("[OPENCV] Ouverture du flux caméra en direct...")
    camera = cv2.VideoCapture(0)

    while True:
        success, frame = camera.read()
        if not success or frame is None:
            print("[OPENCV] Impossible de lire le flux.")
            break

        cv2.imshow("Detection", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

        if cv2.getWindowProperty("Detection", cv2.WND_PROP_VISIBLE) < 1:
            break

    camera.release()
    cv2.destroyAllWindows()
    print("[OPENCV] Fenêtre caméra fermée.")

def activer_camera_flux():
    print(f"\n[OUTIL] Lancement de la caméra...")
    cam_thread = threading.Thread(target=logique_flux_camera, daemon=True)
    cam_thread.start()

@app.get("/", response_class=HTMLResponse)
async def page_accueil():
    return """
    <html>
        <head><title>Accueil Agent IA</title></head>
        <body style="font-family: Arial, sans-serif; text-align: center; margin-top: 50px; background-color: #f4f6f9;">
            <h1 style="color: #2c3e50;">🤖 Bienvenue sur l'Agent IA</h1>
            <p style="font-size: 18px;">Pour lancer directement la détection caméra, clique sur le bouton ci-dessous :</p>
            <br>
            <a href="/demarrer-camera" style="padding: 15px 30px; background-color: #2ecc71; color: white; text-decoration: none; font-weight: bold; border-radius: 5px; font-size: 20px;">🎥 ALLUMER LA CAMÉRA</a>
        </body>
    </html>
    """

@app.get("/demarrer-camera", response_class=HTMLResponse)
async def demarrer_automatique():
    activer_camera_flux()
    return """
    <html>
        <head><title>Détection lancée</title></head>
        <body style="font-family: Arial, sans-serif; text-align: center; margin-top: 50px;">
            <h1 style="color: #2ecc71;">🚀 Détection lancée !</h1>
            <p>La fenêtre de ta webcam vient de s'ouvrir sur ton écran.</p>
            <p>Pour la fermer : clique sur la vidéo et appuie sur la touche <strong>'q'</strong> de ton clavier.</p>
            <br>
            <a href="/" style="color: #3498db;">⬅️ Retour à l'accueil</a>
        </body>
    </html>
    """

if __name__ == "__main__":
    print("\n⚡ LANCEMENT SÉCURISÉ DU SERVEUR SUR JUPYTER...")
    
    # Configuration uvicorn spécifique pour Jupyter (on passe l'application sous forme de chaîne ou d'objet avec loop="none")
    config = uvicorn.Config(app, host="127.0.0.1", port=8090, log_level="info")
    server = uvicorn.Server(config)
    
    # On utilise la boucle d'événements déjà existante de Jupyter au lieu de recréer une boucle avec uvicorn.run()
    import asyncio
    loop = asyncio.get_event_loop()
    loop.create_task(server.serve())


⚡ LANCEMENT SÉCURISÉ DU SERVEUR SUR JUPYTER...


INFO:     Started server process [21132]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8090 (Press CTRL+C to quit)


INFO:     127.0.0.1:55594 - "GET / HTTP/1.1" 200 OK

[OUTIL] Lancement de la caméra...
[OPENCV] Ouverture du flux caméra en direct...
INFO:     127.0.0.1:55597 - "GET /demarrer-camera HTTP/1.1" 200 OK
